# Gumbel approximation of the expected maximum

The Deflated Sharpe Ratio sets its threshold with the expected maximum of N independent standard normals, approximated as (Bailey and López de Prado 2014, eq. 5)

$$E[\max_N] \approx (1-\gamma)\,Z^{-1}\!\left[1-\tfrac{1}{N}\right] + \gamma\,Z^{-1}\!\left[1-\tfrac{1}{Ne}\right]$$

Week 1 check: for N = 10, 100, 1000 simulate the maximum of N standard normals 10 000 times and compare its mean with the formula. The exact value, a numerical integral of the density of the maximum, separates the formula's error from Monte Carlo noise. The paper (Appendix 2, Exhibit 3.1, variance 1) reports an overestimate below 0.05 for N < 50, "as small as 0.006" at N = 1000.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from statarb.dsr import expected_max

SIMS = 10_000
rng = np.random.default_rng(0)
x = np.linspace(-10, 10, 200_001)


def exact_max(n: int) -> float:
    # E[max] = integral of x * n * pdf(x) * cdf(x)^(n-1)
    return float(np.trapezoid(x * n * norm.pdf(x) * norm.cdf(x) ** (n - 1), x))


rows = []
for n in [10, 100, 1000]:
    m = rng.standard_normal((SIMS, n)).max(axis=1)
    se = m.std(ddof=1) / np.sqrt(SIMS)
    rows.append(
        {"N": n, "monte_carlo": m.mean(), "mc_se": se, "exact": exact_max(n), "formula": expected_max(n)}
    )
t = pd.DataFrame(rows).set_index("N")
t["error_vs_mc"] = t["formula"] - t["monte_carlo"]
t["error_vs_exact"] = t["formula"] - t["exact"]
t["error_pct"] = 100 * t["error_vs_exact"] / t["exact"]
t.round(4)

,monte_carlo,mc_se,exact,formula,error_vs_mc,error_vs_exact,error_pct
N,,,,,,,
10,1.5390,0.0059,1.5388,1.5746,0.0356,0.0358,2.3295
100,2.5084,0.0043,2.5076,2.5306,0.0222,0.0230,0.9176
1000,3.2412,0.0035,3.2414,3.2551,0.0139,0.0137,0.4222


In [2]:
assert ((t["monte_carlo"] - t["exact"]).abs() < 4 * t["mc_se"]).all()  # simulation agrees with exact
assert (t["error_vs_exact"] > 0).all()  # the formula overestimates...
assert t["error_vs_exact"].is_monotonic_decreasing  # ...less and less as N grows
assert t.loc[10, "error_vs_exact"] < 0.05  # paper: below 0.05 for N < 50

# The paper's heat map has 101 cells per N (one per mean), each a 10 000-draw simulation with this standard
# error: the smallest cell sits below the true error by about se * E[max of 101 normals].
e, se = t.loc[1000, "error_vs_exact"], t.loc[1000, "mc_se"]
smallest_cell = e - se * expected_max(101)
assert abs(smallest_cell - 0.006) < 2 * se
print(
    f"N = 1000: true error {e:.4f}, MC standard error {se:.4f}, "
    f"expected smallest of 101 noisy cells {smallest_cell:.4f} (paper: 0.006)"
)

N = 1000: true error 0.0137, MC standard error 0.0035, expected smallest of 101 noisy cells 0.0048 (paper: 0.006)


**Result.** The formula overestimates the exact expected maximum by 0.036 at N = 10 (2.3%), 0.023 at N = 100 (0.9%) and 0.014 at N = 1000 (0.4%); the simulation matches the exact value within its standard error. This is in line with the paper: below 0.05 for small N, shrinking as N grows. The paper's 0.006 at N = 1000 is the smallest of 101 noisy heat-map cells (about 0.005 expected here), the error itself is 0.014.

In the DSR the error is scaled by √V. With daily Sharpe ratios over T = 1250 days, √V ≈ 1/√1250 ≈ 0.028, so at N = 100 the threshold SR₀ is too high by about 0.0007 a day (0.01 annualized): the DSR errs on the conservative side, by a negligible amount.